# Fraud Detection - Data Preprocessing

This notebook implements the data preprocessing pipeline based on the insights gathered during the Exploratory Data Analysis (EDA). The goal is to clean the data, handle missing values, engineer relevant features, and prepare the dataset for modeling.


## 1. Import Libraries

We use standard data science libraries for processing and scikit-learn for transformations.


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

## 2. Load and Merge Data

As identified in the EDA, the dataset is split into transaction and identity files. We merge them on `TransactionID` to create a unified dataset.


In [2]:
train_trans = pd.read_csv("../data/raw/train_transaction.csv")
train_id = pd.read_csv("../data/raw/train_identity.csv")

df = train_trans.merge(train_id, on="TransactionID", how="left")
del train_trans, train_id
import gc
gc.collect()

print("Merged Shape:", df.shape)

Merged Shape: (590540, 434)


## 3. Drop Highly Missing Features

During EDA, we observed that many features have a very high percentage of missing values (some over 90%). These columns are unlikely to provide significant predictive power and might introduce noise. We drop features with more than 90% missing values.


In [3]:
missing_percent = df.isnull().mean() * 100

cols_to_drop = missing_percent[missing_percent > 90].index
df.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns with >90% missing values.")
print("After dropping high-missing columns:", df.shape)

Dropped 12 columns with >90% missing values.
After dropping high-missing columns: (590540, 422)


## 4. Feature Engineering

Based on EDA insights, we perform two key transformations on the `TransactionAmt` feature:

1. **Log Transformation**: To handle the high skewness of transaction amounts.
2. **Decimal Extraction**: To capture potential patterns in how transaction values are rounded or entered.


### Log Transform (TransactionAmt)


In [4]:
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

C:\Users\Mohsen\AppData\Local\Temp\ipykernel_7608\1097107937.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])


### Decimal Feature


In [5]:
df['TransactionAmt_decimal'] = (
    (df['TransactionAmt'] - df['TransactionAmt'].astype(int)) * 1000
)


C:\Users\Mohsen\AppData\Local\Temp\ipykernel_7608\914104310.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_decimal'] = (


## 5. Separate Features and Target

We separate the target variable `isFraud` and drop non-predictive identifiers like `TransactionID`.


In [6]:
target = "isFraud"

X = df.drop(columns=[target, "TransactionID"])
y = df[target]


## 6. Handle Missing Values

We apply basic imputation strategies:
- **Numerical Features**: Impute missing values with the median.
- **Categorical Features**: Impute missing values with a placeholder string "Missing".


### Numerical -> Median


In [7]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())


### Categorical -> "Missing"


In [8]:
cat_cols = X.select_dtypes(include=['object', 'str']).columns
X[cat_cols] = X[cat_cols].fillna("Missing")


## 7. Encoding and Scaling

We create two versions of the preprocessed data:
1. **Label Encoded**: Best for tree-based models (XGBoost, LightGBM, Random Forest, Decision Tree).
2. **One-Hot Encoded**: Required for linear models (Logistic Regression).


### 8.1 Label Encoding (on whole dataset)

We apply Label Encoding to the entire dataset before splitting to ensure all categories are captured, following the original notebook's approach.


In [ ]:
X_le = X.copy()
for col in cat_cols:
    le = LabelEncoder()
    X_le[col] = le.fit_transform(X_le[col].astype(str))


### 8.2 Train-Test Split (Stratified)

We perform a stratified split on both the raw data (for OHE) and the label-encoded data.


In [ ]:
# Split for Label Encoded data
X_train, X_test, y_train, y_test = train_test_split(
    X_le, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split for Raw data (to be used for OHE)
X_train_raw, X_test_raw, _, _ = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


### 8.3 Scaling Numerical Features (for Label Encoded data)


In [ ]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


### 8.4 One-Hot Encoding & Scaling (for Logistic Regression)


In [ ]:
# We use ColumnTransformer to One-Hot Encode categorical features and scale numerical ones
# We use max_categories=10 to prevent memory issues in this environment
ct = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=10, dtype=np.float32), cat_cols)
    ],
    remainder='passthrough'
)

X_train_logreg_arr = ct.fit_transform(X_train_raw)
X_test_logreg_arr = ct.transform(X_test_raw)

# Get feature names for the OHE columns
ohe_feature_names = ct.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = list(num_cols) + list(ohe_feature_names)

X_train_logreg = pd.DataFrame(X_train_logreg_arr, columns=all_feature_names)
X_test_logreg = pd.DataFrame(X_test_logreg_arr, columns=all_feature_names)

# Clean up to save memory
del X_train_logreg_arr, X_test_logreg_arr, X_train_raw, X_test_raw
gc.collect()

print("OHE Train Shape:", X_train_logreg.shape)


## 9. Save Processed Data

We save both versions of the processed datasets to the `data/processed` directory.


In [ ]:
os.makedirs("../data/processed", exist_ok=True)

# Save Label Encoded version
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

# Save One-Hot Encoded version
X_train_logreg.to_csv("../data/processed/X_train_logreg.csv", index=False)
X_test_logreg.to_csv("../data/processed/X_test_logreg.csv", index=False)

# Save Target
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Processed datasets saved!")


## 10. Save Transformers

We persist the scaling and encoding objects.


In [ ]:
joblib.dump(scaler, "../data/processed/scaler.pkl")
joblib.dump(ct, "../data/processed/column_transformer_logreg.pkl")
print("Transformers saved!")


## Final Summary of Preprocessing

- **Merging**: Unified Transaction and Identity data.
- **Cleaning**: Removed features with >90% missing values.
- **Engineering**: Applied log transformation to `TransactionAmt` and extracted decimals.
- **Imputation**: Median for numerical, "Missing" for categorical.
- **Encoding**: Label encoding for tree models, One-Hot encoding for linear models.
- **Splitting**: Stratified 80/20 split.
- **Scaling**: Standardized numerical features.

The dataset is now ready for the next stage: **Feature Reduction**.
